# The Efficiency Bridge
## Balancing Gemini 3.1 Flash-Lite vs. Gemini 3.1 Pro

> *A practical guide for architects and senior engineers who want the reasoning power of Pro with the economics of Flash-Lite - by routing intelligently between them.*

---

## Table of Contents

1. [The 'PLRI' Framework](#section-1) - The Principle of Least Required Intelligence
2. [Performance Benchmarking](#section-2) - Side-by-side comparison: cost, reasoning
3. [The 'Stress Test' Prompt](#section-3) - Where Flash-Lite hits its ceiling
4. [The 'Hybrid Router' Pattern](#section-4) - Flash-Lite as triage agent, Pro as specialist
5. [Best Practices for Flash-Lite](#section-5) - 5 techniques to unlock Pro-level results
6. [Full Runnable Demo](#section-6) - End-to-end hybrid system
7. [Quick Reference](#section-7) - Decision matrix and cheat sheet

---
<a id='section-1'></a>
## Section 1: The 'PLRI' Framework
### The Principle of Least Required Intelligence (PLRI)

In production AI, there is a common architectural fallacy: assuming that using the most powerful model is always the safest choice. In reality, over-specifying model power is a form of technical debt that accumulates high latency and unnecessary costs.

The **Principle of Least Required Intelligence (PLRI)** posits that an AI system should always default to the most efficient model capable of handling the specific sub-task at hand.

#### Why PLRI Matters for Scaled Applications:
- **Latency as a Feature:** Shifting 90% of your load from Pro to Flash-Lite can improve response times and create a snappier user experience (measure this in your own stack).
- **Unit Economics:** Gemini 3.1 Flash-Lite is roughly **8x-16x cheaper** than Pro, depending on prompt length and output volume (per Gemini API pricing).
- **Focusing the Expert:** By triaging "lookup" and "extraction" tasks to Flash-Lite, you reserve Pro's specialized reasoning for where it actually moves the needle.

> **The PLRI Rule:** If a task can be solved by an $O(n)$ pattern-matching model (Flash-Lite), do not waste an $O(log n)$ reasoning engine (Pro) on it.

In practice, well-designed Gemini-based systems follow a **90/10 split**:

| Traffic Share | Role | Typical Tasks |
|---|---|---|
| **90% (The Worker)** | Gemini 3.1 Flash-Lite | Classification, entity extraction, summarization, RAG queries, JSON formatting |
| **10% (The Specialist)** | Gemini 3.1 Pro | Complex multi-step reasoning, deep code synthesis, adversarial logic, recursive problem solving |

This guide provides the blueprints for building an **Efficiency Bridge** that applies PLRI automatically.


---
<a id='section-2'></a>
## Section 2: Performance Benchmarking
### Flash-Lite vs. Pro
Before architecting a routing system, you need concrete numbers to justify threshold decisions. The table below reflects published Gemini API pricing and model limits as of March 2026.

#### Cost and Limits

| Metric | Gemini 3.1 Flash-Lite | Gemini 3.1 Pro (<=200k prompt) | Gemini 3.1 Pro (>200k prompt) |
|---|---|---|---|
| **Input cost (per 1M tokens)** | $0.25 | $2.00 | $4.00 |
| **Output cost (per 1M tokens)** | $1.50 | $12.00 | $18.00 |
| **Context window** | 1,048,576 tokens | 1,048,576 tokens | 1,048,576 tokens |
| **Max output tokens** | 65,536 | 65,536 | 65,536 |

*Note: Output pricing includes thinking tokens. Audio input is priced separately.*

#### Reasoning Depth by Thinking Level

| Thinking Level | Flash-Lite Behavior | Pro Behavior | Best Use Case |
|---|---|---|---|
| **`minimal`** | Very fast, likely skips reasoning | Not supported | Formatting, labeling, simple Q&A |
| **`low`** | Light reasoning, very fast | Moderate reasoning | Single-hop logic, classification |
| **`medium`** | Structured reasoning, balanced | Balanced reasoning | Multi-step tasks, code generation |
| **`high`** | Near-Pro on many tasks | Maximum depth | Research, recursive logic, adversarial problems |
| **Default** | Dynamic (`high`) | Dynamic (`high`) | General use |

> **Key insight:** Flash-Lite at `thinking_level="high"` can close part of the gap with Pro on structured tasks, while still being **~8x-16x cheaper** depending on prompt length and output size. The Efficiency Bridge exploits this gap.

#### Task-Level Accuracy (Qualitative)

| Task Category | Flash-Lite | Pro | Recommendation |
|---|---|---|---|
| Entity extraction from text | Excellent | Excellent | Use Flash-Lite |
| Sentiment classification | Excellent | Excellent | Use Flash-Lite |
| Translation (common languages) | Excellent | Excellent | Use Flash-Lite |
| Multi-document summarization | Very Good | Excellent | Use Flash-Lite |
| Single-file code generation | Good | Excellent | Flash-Lite w/ `high` thinking |
| Recursive algorithm design | Fair | Excellent | Route to Pro |
| Multi-file refactoring | Fair | Excellent | Route to Pro |
| Complex math proofs | Poor | Excellent | Always Pro |
| Adversarial / red-team logic | Poor | Excellent | Always Pro |


In [49]:
%pip install google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

FLASH_LITE = "gemini-3.1-flash-lite-preview" 
PRO = "gemini-3.1-pro-preview"

load_dotenv()
client = genai.Client()

---
<a id='section-3'></a>
## Section 3: The 'Stress Test' Prompt
### Where Flash-Lite Hits Its Ceiling - and Why

Understanding *where* a model fails is as important as knowing where it succeeds. Below is a canonical stress-test prompt that reliably separates Flash-Lite from Pro.

#### The Stress-Test Prompt

```
Implement a Python function `flatten_and_deduplicate(data)` that:

1. Accepts an arbitrarily nested structure of lists, tuples, sets, dicts, 
   and generators as input (not just simple lists).
2. Recursively flattens ALL iterable types EXCEPT strings (which must be 
   treated as atomic values, not character sequences).
3. For dict types: flatten only the VALUES, not the keys.
4. Deduplicates the final flat sequence while PRESERVING INSERTION ORDER 
   (i.e., first occurrence wins).
5. Converts all numeric types (int, float, complex) to their string 
   representation BEFORE deduplication, so that 3, 3.0, and 3+0j are 
   treated as DUPLICATES of each other.
6. The function must be lazy: return a generator, not a list.
7. Include a comprehensive test suite using pytest covering:
   - Deeply nested mixed structures (5+ levels)
   - Generators as input
   - The numeric-as-duplicate edge case
   - Empty containers at any nesting depth
   - Strings not being iterated character-by-character
```

#### Why Flash-Lite Struggles Here

This prompt is adversarial in four distinct ways simultaneously:

1. **Recursive depth pressure** - Correctly handling 5+ levels of nesting requires maintaining a mental model of the call stack across the entire generation. Flash-Lite's shorter internal chain-of-thought makes it prone to dropping edge cases (e.g., generators, dict values vs. keys) at deeper recursion levels.

2. **Conflicting constraints that interact** - The "strings are atomic" rule and the "flatten generators" rule appear similar but behave oppositely. Both `str` and `generator` are iterables in Python - the model must track that one is excepted and one is not, simultaneously, across all recursion branches. Flash-Lite on `thinking_level="low"` or `"medium"` will frequently apply the same rule to both.

3. **Cross-constraint deduplication** - The numeric normalization (`3 == 3.0 == 3+0j` as strings) must happen *before* the deduplication set check, and the deduplication must be order-preserving (not a plain `set()`). This requires holding a `seen` set *and* a yielded sequence simultaneously in a working memory model - a pattern that overwhelms smaller context chains.

4. **Lazy evaluation + statefulness** - Returning a generator that maintains a `seen` set across `yield` statements is a non-trivial Python pattern. Flash-Lite often produces an eager list or an incorrect generator that resets state on re-iteration.

**Gemini 3.1 Pro**, with its deep reasoning mode, traces through all constraint interactions explicitly before generating code, producing a correct, well-tested implementation in one shot.

In [44]:
STRESS_TEST_PROMPT = """Implement a Python function flatten_and_deduplicate(data) that:
1. Accepts arbitrarily nested lists, tuples, sets, dicts, and generators.
2. Recursively flattens all iterables EXCEPT strings (treat strings as atomic).
3. For dicts: flatten only the VALUES, not the keys.
4. Deduplicates preserving insertion order (first occurrence wins).
5. Normalize int/float/complex to str BEFORE dedup (so 3, 3.0, 3+0j are duplicates).
6. Return a lazy generator, not a list.
7. Include pytest tests covering 5+ level nesting, generators, numeric duplicates,
   empty containers, and strings not being iterated char-by-char."""

print("Running stress test on Flash-Lite (thinking_level=medium)...")
flash_response = client.models.generate_content(
    model=FLASH_LITE,
    contents=STRESS_TEST_PROMPT,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="medium")
    )
)

print("=== Flash-Lite Output ===")
print(flash_response.text[:2000], "...")

Running stress test on Flash-Lite (thinking_level=medium)...
=== Flash-Lite Output ===
This implementation uses a recursive generator to traverse the data structure and a `seen` set to keep track of normalized values for deduplication.

### The Implementation

```python
import collections.abc

def normalize(val):
    """Normalize numbers to string representation of complex to treat 3, 3.0, 3+0j as identical."""
    if isinstance(val, (int, float, complex)) and not isinstance(val, bool):
        return str(complex(val))
    return val

def _flatten_recursive(data):
    """Helper generator to recursively yield items."""
    # Check if it's a string (atomic) or not iterable
    if isinstance(data, str) or not isinstance(data, collections.abc.Iterable):
        yield data
    elif isinstance(data, dict):
        for v in data.values():
            yield from _flatten_recursive(v)
    else:
        # Handle lists, tuples, sets, generators
        for item in data:
            yield from _fl

In [45]:
print("Running stress test on Pro (default thinking)...")
pro_response = client.models.generate_content(
    model=PRO,
    contents=STRESS_TEST_PROMPT
)

print("=== Pro Output ===")
print(pro_response.text[:2000], "...")

Running stress test on Pro (default thinking)...
=== Pro Output ===
Here is the implementation of `flatten_and_deduplicate` along with the robust `pytest` suite covering all your requirements.

```python
import types
import pytest

def flatten_and_deduplicate(data):
    """
    Recursively flattens arbitrarily nested iterables, dict values, and generators.
    Deduplicates items preserving insertion order. Treats strings as atomic.
    Normalizes int, float, and complex numbers to their string representation 
    before deduplication (e.g., 3, 3.0, 3+0j all become '3').
    Returns a lazy generator.
    """
    seen = set()
    unhashable_seen = []

    def _normalize(x):
        # Exclude booleans from being converted to '1' or '0'
        if isinstance(x, bool):
            return x
            
        # Reduce mathematical equivalents to standard integer representations if applicable
        if isinstance(x, complex):
            if x.imag == 0:
                x = x.real
        i

---
<a id='section-4'></a>
## Section 4: The 'Hybrid Router' Pattern
### Flash-Lite as Triage Agent, Pro as Deep Reasoning Specialist

The Hybrid Router is the core architectural pattern of the Efficiency Bridge. The idea is straightforward:

1. **Every request enters through Flash-Lite** - The triage agent classifies the request's complexity using a structured schema.
2. **Low-complexity requests** are answered in-place by Flash-Lite - fast and cheap.
3. **High-complexity requests** are forwarded to Pro with the original prompt untouched.

The triage call itself is extremely cheap - it only needs to classify complexity, not answer the question. A typical triage call costs under 500 input tokens and under 50 output tokens.

```
REQUEST
  |
  v
[Flash-Lite Triage Agent]
  | Classify: complexity = low | medium | high
  |           task_type = extraction | reasoning | code | creative
  |           estimated_depth = 1..5
  |
  +-- complexity == high OR estimated_depth >= 4  -->  [Gemini 3.1 Pro]
  |
  +-- complexity == medium AND task_type == code  -->  [Flash-Lite, thinking=high]
  |
  +-- everything else  -->  [Flash-Lite, thinking=low or medium]
```

In [41]:
import json
from pydantic import BaseModel, Field
from typing import Literal

# --- Triage schema ---
class TriageDecision(BaseModel):
    complexity: Literal["low", "medium", "high"] = Field(
        description="low=simple lookup or formatting, medium=multi-step task, high=recursive/adversarial/research-grade"
    )
    task_type: Literal["extraction", "classification", "translation", "summarization", "code", "reasoning", "creative"] = Field(
        description="Primary task category"
    )
    estimated_depth: int = Field(
        description="Estimated reasoning chain depth (1=trivial, 5=deep recursive)"
    )
    confidence: float = Field(
        description="Triage confidence score from 0.0 to 1.0"
    )
    reasoning: str = Field(
        description="One sentence explaining the routing decision"
    )

TRIAGE_SYSTEM_PROMPT = """You are a precision routing agent for a two-tier LLM system.
Analyze the user's request and classify its complexity. Be conservative:
if in doubt about whether a coding or logic task is complex, mark it 'high'.
For simple formatting, extraction, or translation, always use 'low'."""

def triage(user_query: str) -> TriageDecision:
    """Use Flash-Lite to classify the complexity of a request."""
    response = client.models.generate_content(
        model=FLASH_LITE,
        contents=[
            types.Content(role="user", parts=[types.Part(text=user_query)])
        ],
        config=types.GenerateContentConfig(
            system_instruction=TRIAGE_SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_schema=TriageDecision,
            thinking_config=types.ThinkingConfig(thinking_level="low")  # Triage is cheap
        )
    )
    try:
        data = json.loads(response.text)
        return TriageDecision(**data)
    except (json.JSONDecodeError, Exception) as e:
        # Default to safe routing: escalate to Pro if triage fails
        print(f"Triage parse error ({e}), defaulting to Pro.")
        return TriageDecision(
            complexity="high", task_type="reasoning",
            estimated_depth=5, confidence=0.0,
            reasoning="Triage failed - safe default to Pro."
        )

def select_model(decision: TriageDecision) -> tuple[str, str]:
    """Map a TriageDecision to a (model_string, thinking_level) pair."""
    if decision.complexity == "high" or decision.estimated_depth >= 4:
        return PRO, "high"
    if decision.complexity == "medium" and decision.task_type == "code":
        return FLASH_LITE, "high"  # Flash-Lite + max thinking for medium code tasks
    if decision.complexity == "medium":
        return FLASH_LITE, "medium"
    return FLASH_LITE, "low"  # Low complexity: fast and cheap

def hybrid_generate(user_query: str) -> str:
    """Full hybrid routing pipeline: triage with Flash-Lite, answer with best model."""
    # Step 1: Triage
    decision = triage(user_query)
    model, thinking_level = select_model(decision)

    print(f"  Triage: complexity={decision.complexity}, task={decision.task_type}, depth={decision.estimated_depth}")
    print(f"  Routing to: {'PRO' if model == PRO else 'FLASH-LITE'} | thinking={thinking_level}")
    print(f"  Reason: {decision.reasoning}")

    # Step 2: Answer with the selected model + thinking level
    response = client.models.generate_content(
        model=model,
        contents=user_query,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level=thinking_level)
        )
    )
    return response.text

print("Hybrid Router ready.")

Hybrid Router ready.


In [32]:
# Test Case 1: Simple task - should route to Flash-Lite, thinking=low
query_simple = "Translate 'Good morning, how are you-' into Spanish."

print("=" * 60)
print(f"QUERY: {query_simple}")
print("-" * 60)
result = hybrid_generate(query_simple)
print(f"ANSWER: {result}")

QUERY: Translate 'Good morning, how are you-' into Spanish.
------------------------------------------------------------
  Triage: complexity=low, task=translation, depth=1
  Routing to: FLASH-LITE | thinking=low
  Reason: The user request is a straightforward language translation task requiring no complex reasoning or multi-step logic.
ANSWER: There are a few ways to say this depending on who you are talking to:

**If you are speaking to someone you know well (informal):**
> "Buenos días, ¿cómo estás?"

**If you are speaking to someone in a formal setting (using 'usted'):**
> "Buenos días, ¿cómo está usted?" (or just "Buenos días, ¿cómo está?")

**If you are speaking to more than one person:**
> "Buenos días, ¿cómo están?"


In [33]:
# Test Case 2: Medium code task - should route to Flash-Lite, thinking=high
query_code = "Write a Python function to implement binary search on a sorted list."

print("=" * 60)
print(f"QUERY: {query_code}")
print("-" * 60)
result = hybrid_generate(query_code)
print(f"ANSWER: {result}")

QUERY: Write a Python function to implement binary search on a sorted list.
------------------------------------------------------------
  Triage: complexity=low, task=code, depth=1
  Routing to: FLASH-LITE | thinking=low
  Reason: Implementing a standard binary search algorithm is a routine coding task that does not require advanced reasoning or multi-step complex logic.
ANSWER: Here is a clean and efficient implementation of the binary search algorithm in Python.

### The Implementation

```python
def binary_search(arr, target):
    """
    Performs a binary search on a sorted list.
    
    Args:
        arr (list): A sorted list of elements.
        target: The value to search for.
        
    Returns:
        int: The index of the target if found, otherwise -1.
    """
    low = 0
    high = len(arr) - 1

    while low <= high:
        # Find the middle index
        mid = (low + high) // 2
        
        # Check if the middle element is the target
        if arr[mid] == target

In [34]:
# Test Case 3: Hard reasoning - should route to Pro
query_hard = """Design and implement a lock-free concurrent queue in Python using 
compare-and-swap semantics. Include a formal proof of linearizability."""

print("=" * 60)
print(f"QUERY: {query_hard[:80]}...")
print("-" * 60)
result = hybrid_generate(query_hard)
print(f"ANSWER (first 500 chars): {result[:500]}...")

QUERY: Design and implement a lock-free concurrent queue in Python using 
compare-and-s...
------------------------------------------------------------
  Triage: complexity=high, task=code, depth=5
  Routing to: PRO | thinking=high
  Reason: The request involves implementing complex lock-free data structures requiring low-level atomic semantics in Python and includes a formal theoretical proof of linearizability.
ANSWER (first 500 chars): To design a lock-free concurrent queue using Compare-And-Swap (CAS) semantics, we use the standard **Michael-Scott Lock-Free Queue** algorithm (1996). 

### A Note on Python and Lock-Free Concurrency
Pure Python does not have native hardware-level atomic pointers exposed in the standard library. Furthermore, the Global Interpreter Lock (GIL) limits true thread-level parallelism for bytecode execution. To demonstrate a mathematically rigorous lock-free algorithm in Python, we must first simulate ...


---
<a id='section-5'></a>
## Section 5: Best Practices for Flash-Lite
### 5 Techniques to Unlock Pro-Level Results

Flash-Lite is not a "dumb" model. It is a focused model. The following techniques consistently close the gap between Flash-Lite and Pro for tasks that fall in the medium-complexity zone.

---

### Tip 1: Set `thinking_level="high"` for Anything Non-Trivial

This is the single highest-return configuration change. Flash-Lite's default thinking level is dynamic (usually `"high"`), but explicitly setting it ensures consistency in production and enables you to A/B test levels.

Use `"high"` for: code generation, multi-step reasoning, structured data extraction with complex schemas, any task where correctness matters more than raw speed.

In [35]:


# TIP 1: Explicitly set thinking_level="high" for complex non-trivial tasks
response = client.models.generate_content(
    model=FLASH_LITE,
    contents="Write a Python decorator that adds retry logic with exponential backoff and jitter.",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="high")
    )
)

print(response.text)

This implementation uses `functools.wraps` to ensure the metadata (like the function name and docstring) of the decorated function is preserved. It also allows you to specify which exceptions should trigger a retry, preventing the decorator from retrying errors that shouldn't be retried (like `ValueError` or `TypeError`).

### The Decorator Implementation

```python
import time
import random
import functools
import logging

# Configure basic logging to see the retries happening
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def retry_with_backoff(max_retries=3, initial_delay=1, backoff_factor=2, exceptions=(Exception,)):
    """
    Decorator that retries a function with exponential backoff and jitter.

    :param max_retries: Maximum number of retry attempts.
    :param initial_delay: Initial delay in seconds.
    :param backoff_factor: Multiplier for the delay for each subsequent retry.
    :param exceptions: A tuple of exception classes to catch and re

---

### Tip 2: Use Few-Shot Prompting to Anchor Output Format and Reasoning Style

Few-shot examples are dramatically more effective with Flash-Lite than with Pro, because Flash-Lite relies more heavily on pattern-matching from examples rather than abstract instruction-following. **Provide 2-3 input/output pairs** that demonstrate the exact output format and reasoning style you expect. This alone can lift accuracy on structured tasks by 15-30%.

In [36]:


# TIP 2: Few-shot prompting to anchor classification format
few_shot_prompt = """Classify each SQL query as: SELECT, INSERT, UPDATE, DELETE, or DDL.

Example 1:
Query: SELECT name, age FROM users WHERE active = 1;
Classification: SELECT

Example 2:
Query: INSERT INTO orders (user_id, product_id, qty) VALUES (42, 7, 3);
Classification: INSERT

Example 3:
Query: ALTER TABLE products ADD COLUMN discount_pct FLOAT DEFAULT 0;
Classification: DDL

Now classify:
Query: UPDATE inventory SET stock = stock - 1 WHERE product_id = 99;
Classification:"""

response = client.models.generate_content(
    model=FLASH_LITE,
    contents=few_shot_prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="low")  # Even low thinking is fine with good few-shot
    )
)

print(response.text)

Classification: UPDATE


---

### Tip 3: Force Structured JSON Output with a Pydantic Schema

Unconstrained generation is where Flash-Lite is most likely to drift - adding unwanted commentary, changing output format, or hallucinating fields. **Locking the output to a Pydantic schema via `response_schema`** eliminates an entire class of failures. Parse errors drop to near zero, and downstream code becomes trivial.

In [37]:
import json
from pydantic import BaseModel, Field
from typing import List

# TIP 3: Structured JSON output with schema enforcement
class CodeReview(BaseModel):
    overall_quality: str = Field(description="poor, fair, good, or excellent")
    issues: List[str] = Field(description="List of specific issues found")
    suggestions: List[str] = Field(description="List of concrete improvement suggestions")
    security_concerns: bool = Field(description="True if any security issues detected")
    estimated_complexity: str = Field(description="O(1), O(n), O(n^2), etc.")

code_snippet = """
def find_user(users, target_id):
    for i in range(len(users)):
        if users[i]['id'] == target_id:
            return users[i]
    return None
"""

response = client.models.generate_content(
    model=FLASH_LITE,
    contents=f"Review this Python code:\n{code_snippet}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=CodeReview,
        thinking_config=types.ThinkingConfig(thinking_level="medium")
    )
)

try:
    review = json.loads(response.text)
    print(json.dumps(review, indent=2))
except json.JSONDecodeError as e:
    print(f"Parse error: {e}")
    print(response.text)

{
  "overall_quality": "fair",
  "issues": [
    "Linear search is inefficient for large datasets",
    "Uses indexing with range(len()) which is non-idiomatic Python",
    "Potential KeyError if an entry lacks the 'id' key"
  ],
  "suggestions": [
    "Iterate directly over elements using 'for user in users:'",
    "Use a dictionary for O(1) lookups if the list is searched frequently",
    "Use 'user.get('id')' to safely handle missing keys"
  ],
  "security_concerns": false,
  "estimated_complexity": "O(n)"
}


---

### Tip 4: Use System Instructions to Constrain Scope and Persona

Flash-Lite is extremely responsive to system-level constraints. A well-crafted system instruction that **eliminates unnecessary output**, defines a precise persona, and sets hard boundaries ("return only the answer, no commentary") reduces hallucination surface area dramatically. Think of system instructions as guardrails that prevent the model from expending tokens on things you don't need.

In [38]:


# TIP 4: Tightly scoped system instruction
system_instruction = """You are a data extraction engine with zero tolerance for ambiguity.
Rules:
- Extract ONLY the requested fields. Never add commentary, caveats, or explanation.
- If a field is not present in the input, return null for that field.
- If a date is ambiguous, prefer ISO 8601 format (YYYY-MM-DD).
- Never infer or guess values that are not explicitly stated in the input."""

raw_text = """
Hi, I placed an order last Tuesday for 3 units of the Pro Wireless Headset (SKU: HW-2240).
My account is under john.doe@example.com. The order confirmation number is ORD-88721.
It still hasn't shipped and I'm pretty frustrated.
"""

response = client.models.generate_content(
    model=FLASH_LITE,
    contents=f"Extract: customer_email, order_id, sku, quantity, issue_type.\n\nInput: {raw_text}",
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json"
    )
)

print(response.text)

{
  "customer_email": "john.doe@example.com",
  "order_id": "ORD-88721",
  "sku": "HW-2240",
  "quantity": 3,
  "issue_type": "delayed shipping"
}


---

### Tip 5: Use Search Grounding to Eliminate Hallucination on Facts

For tasks requiring current knowledge - API versions, pricing, recent events, latest model specs - Flash-Lite without grounding will confidently hallucinate. **Enable `google_search` grounding** to anchor responses in real retrieved data. This is the fastest way to make Flash-Lite reliable on time-sensitive, knowledge-intensive questions without routing to Pro.

In [39]:


# TIP 5: Search grounding to prevent hallucination on facts
google_search_tool = types.Tool(google_search=types.GoogleSearch())

response = client.models.generate_content(
    model=FLASH_LITE,
    contents="What is the current pricing for Gemini 3.1 Flash-Lite and Gemini 3.1 Pro on the Gemini API?",
    config=types.GenerateContentConfig(
        tools=[google_search_tool],
        thinking_config=types.ThinkingConfig(thinking_level="low")  # Grounding does the heavy lifting
    )
)

print(response.text)

# Check grounding sources used
if response.candidates[0].grounding_metadata:
    gm = response.candidates[0].grounding_metadata
    print(f"\nGrounded against {len(gm.grounding_chunks)} source(s).")

As of March 2026, the Gemini 3.1 models are available in preview through the Gemini API. Below is the pricing information for Gemini 3.1 Pro and Gemini 3.1 Flash-Lite.

### **Gemini 3.1 Pro**
Gemini 3.1 Pro is priced based on the size of your input/output prompts.

*   **Input Pricing:**
    *   **Prompts ≤ 200k tokens:** $2.00 per 1 million tokens
    *   **Prompts > 200k tokens:** $4.00 per 1 million tokens
*   **Output Pricing:**
    *   **Prompts ≤ 200k tokens:** $12.00 per 1 million tokens
    *   **Prompts > 200k tokens:** $18.00 per 1 million tokens

### **Gemini 3.1 Flash-Lite**
Gemini 3.1 Flash-Lite is designed for high-volume, cost-efficient workloads and uses a simpler pricing structure:

*   **Input Pricing:** $0.25 per 1 million tokens
*   **Output Pricing:** $1.50 per 1 million tokens

***

*Note: Pricing and availability can change, especially for models in "Preview" status. It is recommended to check the official [Google AI Studio pricing page](https://aistudio.google.c

---
<a id='section-6'></a>
## Section 6: Full Runnable Demo
### End-to-End Hybrid System with Cost Tracking

This demo wires the triage router into a full request loop and tracks token usage and estimated cost per request.

In [40]:
import json
from dataclasses import dataclass, field
from typing import Literal
from pydantic import BaseModel, Field as PField

# Approximate cost per 1M tokens (USD) as of March 2026 (see Gemini API pricing)
COST_TABLE = {
    FLASH_LITE: {"input": 0.25, "output": 1.50},
}
PRO_PRICING = {
    "le_200k": {"input": 2.00, "output": 12.00},
    "gt_200k": {"input": 4.00, "output": 18.00},
}

@dataclass
class RequestResult:
    query: str
    model_used: str
    thinking_level: str
    triage_complexity: str
    input_tokens: int
    output_tokens: int
    estimated_cost_usd: float
    answer: str

class TriageDecision(BaseModel):
    complexity: Literal["low", "medium", "high"]
    task_type: Literal["extraction", "classification", "translation", "summarization", "code", "reasoning", "creative"]
    estimated_depth: int
    confidence: float
    reasoning: str

TRIAGE_SYSTEM = """You are a routing agent for a two-tier LLM system. Classify the request complexity.
Be conservative: default to 'high' when facing recursive logic, proofs, or adversarial tasks."""

def run_hybrid_request(user_query: str) -> RequestResult:
    # --- Triage (always Flash-Lite, minimal thinking) ---
    triage_resp = client.models.generate_content(
        model=FLASH_LITE,
        contents=user_query,
        config=types.GenerateContentConfig(
            system_instruction=TRIAGE_SYSTEM,
            response_mime_type="application/json",
            response_schema=TriageDecision,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")
        )
    )

    try:
        decision = TriageDecision(**json.loads(triage_resp.text))
    except Exception:
        decision = TriageDecision(
            complexity="high", task_type="reasoning",
            estimated_depth=5, confidence=0.0,
            reasoning="Triage parse failed - routing to Pro."
        )

    # --- Routing decision ---
    if decision.complexity == "high" or decision.estimated_depth >= 4:
        model, thinking = PRO, "high"
    elif decision.complexity == "medium" and decision.task_type == "code":
        model, thinking = FLASH_LITE, "high"
    elif decision.complexity == "medium":
        model, thinking = FLASH_LITE, "medium"
    else:
        model, thinking = FLASH_LITE, "low"

    # --- Main generation ---
    main_resp = client.models.generate_content(
        model=model,
        contents=user_query,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level=thinking)
        )
    )

    # --- Cost calculation ---
    usage = main_resp.usage_metadata
    in_tok  = usage.prompt_token_count or 0
    out_tok = usage.candidates_token_count or 0
    if model == PRO:
        tier = "gt_200k" if in_tok > 200_000 else "le_200k"
        costs = PRO_PRICING[tier]
    else:
        costs = COST_TABLE[model]
    estimated_cost = (in_tok * costs["input"] + out_tok * costs["output"]) / 1_000_000

    return RequestResult(
        query=user_query[:60],
        model_used=model.split("-preview")[0],  # Trim for display
        thinking_level=thinking,
        triage_complexity=decision.complexity,
        input_tokens=in_tok,
        output_tokens=out_tok,
        estimated_cost_usd=estimated_cost,
        answer=main_resp.text
    )

# Run a batch of mixed-complexity requests
test_queries = [
    "What is the capital of France?",
    "Translate 'I need help with my invoice' into Japanese.",
    "Write a Python function to merge two sorted linked lists.",
    "Prove that the square root of 2 is irrational using proof by contradiction.",
]

total_cost = 0.0
print(f"{'Query':<45} {'Model':<25} {'Thinking':<10} {'In Tok':>6} {'Out Tok':>7} {'Cost(USD)':>10}")
print("-" * 110)

for q in test_queries:
    result = run_hybrid_request(q)
    total_cost += result.estimated_cost_usd
    print(
        f"{result.query:<45} {result.model_used:<25} {result.thinking_level:<10} "
        f"{result.input_tokens:>6} {result.output_tokens:>7} ${result.estimated_cost_usd:.6f}"
    )

print("-" * 110)
print(f"{'Total estimated cost for this batch:':<90} ${total_cost:.6f}")

Query                                         Model                     Thinking   In Tok Out Tok  Cost(USD)
--------------------------------------------------------------------------------------------------------------
What is the capital of France?                gemini-3.1-flash-lite     low             8       7 $0.000013
Translate 'I need help with my invoice' into Japanese. gemini-3.1-flash-lite     low            13     292 $0.000441
Write a Python function to merge two sorted linked lists. gemini-3.1-flash-lite     high           12     759 $0.001141
Prove that the square root of 2 is irrational using proof by gemini-3.1-pro            high           16     797 $0.009596
--------------------------------------------------------------------------------------------------------------
Total estimated cost for this batch:                                                       $0.011191


---
<a id='section-7'></a>
## Section 7: Quick Reference
### The Efficiency Bridge - Decision Matrix and Cheat Sheet

#### Routing Decision Matrix

| Task Type | Examples | Recommended Model | Thinking Level |
|---|---|---|---|
| Simple lookup / Q&A | Capital cities, unit conversions | Flash-Lite | `low` |
| Extraction | Entity extraction, form parsing | Flash-Lite | `low` |
| Classification | Sentiment, category routing | Flash-Lite | `low` |
| Translation | Any language pair | Flash-Lite | `low` |
| Summarization | Documents, emails, threads | Flash-Lite | `medium` |
| Simple code | Utility functions, CRUD | Flash-Lite | `high` |
| Multi-step code | Decorators, design patterns | Flash-Lite | `high` |
| Complex algorithms | Graph traversal, DP solutions | Pro | `high` |
| Recursive logic | Tree algorithms, parsers | Pro | `high` |
| Mathematical proofs | Formal proofs, theorem proving | Pro | `high` |
| Multi-file refactoring | Architecture changes | Pro | `high` |
| Adversarial / red-team | Security analysis, edge cases | Pro | `high` |
| Research-grade analysis | Literature synthesis, PhD-level | Pro | `high` |

#### Key Constants

| Constant | Value |
|---|---|
| Flash-Lite model string | `gemini-3.1-flash-lite-preview` |
| Pro model string | `gemini-3.1-pro-preview` |
| Flash-Lite input cost | $0.25 / 1M tokens |
| Flash-Lite output cost | $1.50 / 1M tokens |
| Pro input cost (<=200k prompt) | $2.00 / 1M tokens |
| Pro output cost (<=200k prompt) | $12.00 / 1M tokens |
| Pro input cost (>200k prompt) | $4.00 / 1M tokens |
| Pro output cost (>200k prompt) | $18.00 / 1M tokens |
| Cost ratio | ~8x-16x cheaper on Flash-Lite (prompt-length dependent) |
| Target traffic split | 90% Flash-Lite / 10% Pro |

#### The 5 Flash-Lite Power Moves

| Tip | When to Use | Impact |
|---|---|---|
| Set `thinking_level="high"` | Any complex or correctness-critical task | High |
| Few-shot examples (2-3 pairs) | Classification, extraction, formatting | High |
| `response_schema=PydanticModel` | Any structured output | Critical |
| Tight system instruction | All production use cases | High |
| Search grounding | Any time-sensitive fact lookup | Critical |
